<a href="https://colab.research.google.com/github/simar-rekhi/leskovec/blob/main/gcn_mutag_hyperop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

GCN / MUTAG with Hyperparam Tuning

In [1]:
# --- Install PyTorch (usually already present on Colab, but safe) ---
!pip -q install --upgrade pip
!pip -q install torch torchvision torchaudio

# --- Install PyTorch Geometric using the correct wheel index for your Torch build ---
import torch, re

ver = torch.__version__              # e.g. "2.3.0+cu121" or "2.3.0+cpu"
m = re.match(r"(\d+\.\d+\.\d+)\+(.*)", ver)
if not m:
    raise RuntimeError(f"Unexpected torch version format: {ver}")

torch_ver, build = m.group(1), m.group(2)
wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+{build}.html"

print("Torch version:", ver)
print("Using PyG wheel index:", wheel_url)

!pip -q install torch-geometric -f {wheel_url}

# Optional sanity print
import torch_geometric
print("PyG version:", torch_geometric.__version__)

Torch version: 2.10.0+cpu
Using PyG wheel index: https://data.pyg.org/whl/torch-2.10.0+cpu.html
PyG version: 2.7.0


In [2]:
import os
import random
import itertools
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

In [3]:
@dataclass
class TrainConfig:
    dataset: str = "MUTAG"
    root: str = "./data/TUDataset"

    # training
    batch_size: int = 32
    lr: float = 1e-3
    weight_decay: float = 5e-4
    epochs: int = 50

    # model
    hidden_dim: int = 64
    num_layers: int = 3
    dropout: float = 0.5

    # split + reproducibility
    seed: int = 42

    # device
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [4]:
def load_dataset(cfg: TrainConfig) -> TUDataset:
    return TUDataset(root=cfg.root, name=cfg.dataset)


def split_dataset_60_10_30(dataset, seed: int = 42):
    """
    60% train, 10% val, 30% test
    """
    indices = list(range(len(dataset)))
    rng = random.Random(seed)
    rng.shuffle(indices)

    n = len(indices)
    n_train = int(0.60 * n)
    n_val   = int(0.10 * n)
    # remainder goes to test (≈30%)
    train_idx = indices[:n_train]
    val_idx   = indices[n_train:n_train + n_val]
    test_idx  = indices[n_train + n_val:]

    train_set = [dataset[i] for i in train_idx]
    val_set   = [dataset[i] for i in val_idx]
    test_set  = [dataset[i] for i in test_idx]
    return train_set, val_set, test_set


def make_loaders_3(train_set, val_set, test_set, batch_size: int):
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_set, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

In [5]:
class GCNGraphClassifier(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, num_layers: int, dropout: float):
        super().__init__()
        if num_layers < 2:
            raise ValueError("num_layers must be >= 2")

        self.dropout = dropout
        self.convs = nn.ModuleList()

        self.convs.append(GCNConv(in_dim, hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
        self.convs.append(GCNConv(hidden_dim, hidden_dim))

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x, edge_index, batch):
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        g = global_mean_pool(x, batch)  # graph embeddings
        return self.classifier(g)

In [6]:
def build_model(dataset: TUDataset, cfg: TrainConfig) -> nn.Module:
    in_dim = dataset.num_features
    out_dim = dataset.num_classes

    if in_dim == 0:
        raise ValueError(
            "This dataset has dataset.num_features == 0.\n"
            "MUTAG usually has features; if you switched datasets, you may need to add features."
        )

    return GCNGraphClassifier(
        in_dim=in_dim,
        hidden_dim=cfg.hidden_dim,
        out_dim=out_dim,
        num_layers=cfg.num_layers,
        dropout=cfg.dropout,
    )

In [7]:
def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer, device: str) -> float:
    model.train()
    total_loss = 0.0
    total_graphs = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        logits = model(batch.x, batch.edge_index, batch.batch)
        loss = F.cross_entropy(logits, batch.y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs
        total_graphs += batch.num_graphs

    return total_loss / max(total_graphs, 1)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: str) -> float:
    model.eval()
    correct = 0
    total = 0

    for batch in loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.batch)
        preds = logits.argmax(dim=-1)
        correct += int((preds == batch.y).sum().item())
        total += batch.num_graphs

    return correct / max(total, 1)

In [8]:
HYPERPARAM_GRID = {
    "lr": [1e-3, 5e-4],
    "weight_decay": [5e-4, 1e-4],
    "hidden_dim": [32, 64, 128],
    "num_layers": [2, 3, 4],
    "dropout": [0.2, 0.5],
    "batch_size": [16, 32],
    "epochs": [50],  # keep fixed during tuning; adjust later if you want
}

def grid_configs(grid: dict):
    keys = list(grid.keys())
    values = [grid[k] for k in keys]
    for combo in itertools.product(*values):
        yield dict(zip(keys, combo))

In [9]:
def run_trial(base_cfg: TrainConfig, trial_params: dict, dataset: TUDataset) -> float:
    """
    Train on train split, select best by val accuracy.
    Returns best val accuracy achieved during the run.
    """
    cfg = TrainConfig(**{**base_cfg.__dict__, **trial_params})
    set_seed(cfg.seed)

    train_set, val_set, test_set = split_dataset_60_10_30(dataset, seed=cfg.seed)
    train_loader, val_loader, _ = make_loaders_3(train_set, val_set, test_set, cfg.batch_size)

    model = build_model(dataset, cfg).to(cfg.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_val = 0.0
    for _epoch in range(1, cfg.epochs + 1):
        _loss = train_one_epoch(model, train_loader, optimizer, cfg.device)
        val_acc = evaluate(model, val_loader, cfg.device)
        if val_acc > best_val:
            best_val = val_acc

    return best_val


def hyperparameter_search(base_cfg: TrainConfig, dataset: TUDataset, grid: dict, max_trials: int = None):
    trials = list(grid_configs(grid))
    if max_trials is not None:
        trials = trials[:max_trials]

    best_params = None
    best_val = -1.0

    print(f"Running {len(trials)} hyperparameter trials...")

    for i, params in enumerate(trials, 1):
        val_acc = run_trial(base_cfg, params, dataset)
        print(f"Trial {i:02d}/{len(trials)} | best_val={val_acc:.3f} | params={params}")

        if val_acc > best_val:
            best_val = val_acc
            best_params = params

    return best_params, best_val

In [10]:
def train_with_best_params(base_cfg: TrainConfig, dataset: TUDataset, best_params: dict):
    cfg = TrainConfig(**{**base_cfg.__dict__, **best_params})
    set_seed(cfg.seed)

    train_set, val_set, test_set = split_dataset_60_10_30(dataset, seed=cfg.seed)
    train_loader, val_loader, test_loader = make_loaders_3(train_set, val_set, test_set, cfg.batch_size)

    model = build_model(dataset, cfg).to(cfg.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_val = 0.0
    best_test_at_best_val = 0.0

    for epoch in range(1, cfg.epochs + 1):
        loss = train_one_epoch(model, train_loader, optimizer, cfg.device)
        val_acc = evaluate(model, val_loader, cfg.device)
        test_acc = evaluate(model, test_loader, cfg.device)

        if val_acc > best_val:
            best_val = val_acc
            best_test_at_best_val = test_acc

        if epoch == 1 or epoch % 10 == 0 or epoch == cfg.epochs:
            print(f"Epoch {epoch:03d}/{cfg.epochs} | loss={loss:.4f} | val={val_acc:.3f} | test={test_acc:.3f}")

    print("\n=== Best Hyperparams (by val) ===")
    print(best_params)
    print(f"Best val acc: {best_val:.3f}")
    print(f"Test acc @ best val: {best_test_at_best_val:.3f}")

    return cfg, model

In [11]:
base_cfg = TrainConfig(
    dataset="MUTAG",
    epochs=50,     # baseline; HPO overrides if grid includes epochs
    seed=42,
)

dataset = load_dataset(base_cfg)
print(f"Loaded {base_cfg.dataset}: graphs={len(dataset)} classes={dataset.num_classes} features={dataset.num_features}")

# Tune (limit trials while debugging; set to None to run full grid)
best_params, best_val = hyperparameter_search(base_cfg, dataset, HYPERPARAM_GRID, max_trials=20)

# Final run using best params
final_cfg, final_model = train_with_best_params(base_cfg, dataset, best_params)
print("\nFinal config used:\n", final_cfg)

Processing...
Done!


Loaded MUTAG: graphs=188 classes=2 features=7
Running 20 hyperparameter trials...
Trial 01/20 | best_val=0.778 | params={'lr': 0.001, 'weight_decay': 0.0005, 'hidden_dim': 32, 'num_layers': 2, 'dropout': 0.2, 'batch_size': 16, 'epochs': 50}
Trial 02/20 | best_val=0.778 | params={'lr': 0.001, 'weight_decay': 0.0005, 'hidden_dim': 32, 'num_layers': 2, 'dropout': 0.2, 'batch_size': 32, 'epochs': 50}
Trial 03/20 | best_val=0.778 | params={'lr': 0.001, 'weight_decay': 0.0005, 'hidden_dim': 32, 'num_layers': 2, 'dropout': 0.5, 'batch_size': 16, 'epochs': 50}
Trial 04/20 | best_val=0.778 | params={'lr': 0.001, 'weight_decay': 0.0005, 'hidden_dim': 32, 'num_layers': 2, 'dropout': 0.5, 'batch_size': 32, 'epochs': 50}
Trial 05/20 | best_val=0.778 | params={'lr': 0.001, 'weight_decay': 0.0005, 'hidden_dim': 32, 'num_layers': 3, 'dropout': 0.2, 'batch_size': 16, 'epochs': 50}
Trial 06/20 | best_val=0.778 | params={'lr': 0.001, 'weight_decay': 0.0005, 'hidden_dim': 32, 'num_layers': 3, 'dropout': 0